# Heart Failure Prediction Model - Enhanced Version

This notebook integrates the `CardioDataSets` data, implements an improved heart failure prediction model, and provides advanced evaluation metrics including AUPRC and specialized risk graphs.

In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install torch pandas numpy matplotlib seaborn scikit-learn tqdm joblib

In [ ]:
# 2. IMPORTS
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, average_precision_score, 
    confusion_matrix, classification_report, brier_score_loss, 
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.calibration import calibration_curve
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 3. DATA PREPARATION

In [ ]:
# Download and prepare dataset
!curl -L -o cardiac_failure_data.csv "https://archive.ics.uci.edu/ml/machine-learning-databases/00519/cardiac_failure_data.csv"

def prepare_data():
    df = pd.read_csv('cardiac_failure_data.csv')
    
    def create_multi_horizon_labels(df):
        labels = np.zeros((len(df), 3))
        for i in range(len(df)):
            labels[i, 2] = df.iloc[i]['DEATH_EVENT']
            if labels[i, 2] == 1:
                high_risk = (df.iloc[i]['serum_creatinine'] > 1.5 or df.iloc[i]['ejection_fraction'] < 30)
                labels[i, 1] = 1 if high_risk else np.random.choice([0, 1])
            if labels[i, 1] == 1:
                labels[i, 0] = 1 if df.iloc[i]['serum_creatinine'] > 2.0 else np.random.choice([0, 1])
        return labels

    def simulate_vitals(row, seq_len=48):
        ef_factor = row['ejection_fraction'] / 100
        base_hr = 70 + (1 - ef_factor) * 30
        hr = np.random.normal(base_hr, 8, seq_len)
        base_bp = 110 + row['high_blood_pressure'] * 25
        bp = np.random.normal(base_bp, 12, seq_len)
        base_spo2 = 95 - (1 - ef_factor) * 8
        spo2 = np.random.normal(base_spo2, 2, seq_len)
        return np.stack([hr, bp, spo2], axis=1)

    labels = create_multi_horizon_labels(df)
    vitals = np.array([simulate_vitals(df.iloc[i]) for i in range(len(df))])
    static = df.drop('DEATH_EVENT', axis=1).values.astype(np.float32)
    
    return df, vitals, static, labels

df, vitals, static, labels = prepare_data()
print("Data loaded and pre-processed.")

# 4. MODEL ARCHITECTURE

In [ ]:
class ImprovedChronosModel(nn.Module):
    def __init__(self, ts_input=3, static_input=11, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(ts_input, hidden_dim, batch_first=True, dropout=dropout)
        self.static_net = nn.Sequential(
            nn.Linear(static_input, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.7)
        )
        self.head_1d = nn.Linear(hidden_dim, 1)
        self.head_7d = nn.Linear(hidden_dim, 1)
        self.head_30d = nn.Linear(hidden_dim, 1)

    def forward(self, x_ts, x_static):
        _, h_n = self.gru(x_ts)
        h_ts = h_n[-1]
        h_static = self.static_net(x_static)
        combined = torch.cat([h_ts, h_static], dim=1)
        hidden = self.classifier(combined)
        y_1d = torch.sigmoid(self.head_1d(hidden))
        y_7d = torch.sigmoid(self.head_7d(hidden))
        y_30d = torch.sigmoid(self.head_30d(hidden))
        return torch.cat([y_1d, y_7d, y_30d], dim=1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
        self.bce = nn.BCELoss(reduction='none')

    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

# 5. EVALUATION UTILITIES

In [ ]:
def compute_all_metrics(y_true_list, y_pred_list, horizons=['1-Day', '7-Day', '30-Day']):
    all_metrics = []
    for i, horizon in enumerate(horizons):
        y_true, y_pred = y_true_list[:, i], y_pred_list[:, i]
        y_pred_binary = (y_pred >= 0.5).astype(int)
        roc_auc = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
        
        metrics = {
            'Horizon': horizon,
            'Accuracy': accuracy_score(y_true, y_pred_binary),
            'Precision': precision_score(y_true, y_pred_binary, zero_division=0),
            'Recall': recall_score(y_true, y_pred_binary, zero_division=0),
            'F1-Score': f1_score(y_true, y_pred_binary, zero_division=0),
            'ROC-AUC': roc_auc,
            'AUPRC': average_precision_score(y_true, y_pred),
            'Brier': brier_score_loss(y_true, y_pred)
        }
        all_metrics.append(metrics)
    return pd.DataFrame(all_metrics)

def plot_evaluation_dashboard(y_true_list, y_pred_list, horizons=['1-Day', '7-Day', '30-Day']):
    # 1. Metrics Table
    metrics_df = compute_all_metrics(y_true_list, y_pred_list, horizons)
    print("\n--- PERFORMANCE METRICS ---")
    print(metrics_df.to_string(index=False))
    
    # 2. Precision-Recall Curves
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    for i, horizon in enumerate(horizons):
        p, r, _ = precision_recall_curve(y_true_list[:, i], y_pred_list[:, i])
        auprc = average_precision_score(y_true_list[:, i], y_pred_list[:, i])
        plt.plot(r, p, label=f'{horizon} (AUPRC: {auprc:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves')
    plt.legend()
    plt.grid(True)
    
    # 3. Calibration Curves
    plt.subplot(1, 2, 2)
    for i, horizon in enumerate(horizons):
        prob_true, prob_pred = calibration_curve(y_true_list[:, i], y_pred_list[:, i], n_bins=10)
        plt.plot(prob_pred, prob_true, marker='o', label=f'{horizon}')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives')
    plt.title('Calibration Curves')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_specialized_auprc(y_true, y_pred, horizon='30-Day'):
    p, r, _ = precision_recall_curve(y_true, y_pred)
    auprc = average_precision_score(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    plt.fill_between(r, p, alpha=0.2, color='green')
    plt.plot(r, p, color='green', lw=2, label=f'AUPRC = {auprc:.4f}')
    plt.axhline(y=y_true.mean(), linestyle='--', color='red', label=f'No Skill ({y_true.mean():.2f})')
    plt.xlabel('Recall (Sensitivity)')
    plt.ylabel('Precision (Positive Predictive Value)')
    plt.title(f'Specialized AUPRC Graph - {horizon} Risk')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.show()

# 6. TRAINING

In [ ]:
# Dataset and Dataloaders
indices = np.arange(len(df))
train_val_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_idx, val_idx = train_test_split(train_val_idx, test_size=0.2, random_state=42)

scaler_v = StandardScaler()
v_train = scaler_v.fit_transform(vitals[train_idx].reshape(-1, 3)).reshape(-1, 48, 3)
v_val = scaler_v.transform(vitals[val_idx].reshape(-1, 3)).reshape(-1, 48, 3)
v_test = scaler_v.transform(vitals[test_idx].reshape(-1, 3)).reshape(-1, 48, 3)

scaler_s = StandardScaler()
s_train = scaler_s.fit_transform(static[train_idx])
s_val = scaler_s.transform(static[val_idx])
s_test = scaler_s.transform(static[test_idx])

class HFD(Dataset):
    def __init__(self, v, s, l):
        self.v, self.s, self.l = torch.FloatTensor(v), torch.FloatTensor(s), torch.FloatTensor(l)
    def __len__(self): return len(self.v)
    def __getitem__(self, idx): return self.v[idx], self.s[idx], self.l[idx]

train_loader = DataLoader(HFD(v_train, s_train, labels[train_idx]), batch_size=32, shuffle=True)
val_loader = DataLoader(HFD(v_val, s_val, labels[val_idx]), batch_size=32)
test_loader = DataLoader(HFD(v_test, s_test, labels[test_idx]), batch_size=32)

model = ImprovedChronosModel(static_input=static.shape[1]).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = FocalLoss()

for epoch in range(10):
    model.train()
    t_loss = 0
    for v, s, l in train_loader:
        v, s, l = v.to(device), s.to(device), l.to(device)
        optimizer.zero_grad()
        out = model(v, s)
        loss = criterion(out, l)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()
    print(f"Epoch {epoch+1}/10 - Loss: {t_loss/len(train_loader):.4f}")

# 7. COMPREHENSIVE EVALUATION

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for v, s, l in test_loader:
        v, s, l = v.to(device), s.to(device), l.to(device)
        all_preds.append(model(v, s).cpu().numpy())
        all_labels.append(l.cpu().numpy())

preds = np.vstack(all_preds)
labs = np.vstack(all_labels)

# 1. Multi-horizon Dashboard
plot_evaluation_dashboard(labs, preds)

# 2. Specialized AUPRC Graph for 30-Day Risk
plot_specialized_auprc(labs[:, 2], preds[:, 2], horizon='30-Day')

# 8. EXTRACT WEIGHTS & SCALERS

In [ ]:
print("--- 30-Day Prediction Head Weights (Top 10) ---")
print(model.head_30d.weight.detach().cpu().numpy().flatten()[:10])

print("\n--- Scaler Parameters (Static Features Highlight) ---")
feat_names = df.drop('DEATH_EVENT', axis=1).columns.tolist()
for i, name in enumerate(feat_names[:5]):
    print(f"{name}: Mean={scaler_s.mean_[i]:.4f}, Scale={scaler_s.scale_[i]:.4f}")

# 9. SAVE MODEL & SCALERS

In [ ]:
import joblib

# 1. Save the PyTorch Model Weights
MODEL_FILENAME = 'improved_chronos_model.pth'
torch.save(model.state_dict(), MODEL_FILENAME)
print(f"Model weights saved to {MODEL_FILENAME}")

# 2. Save the Scalers 
joblib.dump(scaler_v, 'vitals_scaler.pkl')
joblib.dump(scaler_s, 'static_scaler.pkl')
print("Vitals and Static scalers saved as .pkl files")